# **Regression: Linear Regression**

## **Justification of Preprocessing Strategy**

### **Scale Invariance vs. Coefficient Interpretation**
Ordinary Least Squares (OLS) **Linear Regression** is mathematically scale-invariant regarding its final predictions. This means that whether the data is raw, standardized, or normalized, the model will output the exact same predictions. However, the magnitude and scale of the learned coefficients ($\beta$ weights) depend entirely on the scale of their corresponding features. To ensure that we can directly compare the clinical importance of each feature on the `diabetes_risk_score` (interpreting which variable has the strongest impact), we will evaluate both **Standardization** and **Normalization** to find the most numerically stable representation.

### **Data Integrity and Leakage Prevention**
To successfully shift our objective from classification to regression, we strictly drop the previous classification targets (`diagnosed_diabetes` and `diabetes_stage`) to avoid any data leakage. Furthermore, because the target variable `diabetes_risk_score` is continuous, we **omit the stratification parameter** during the train-test split, as stratification is mathematically exclusive to categorical classes.


## **Experiment Design**

We have designed a tournament of **2 focused runs** to establish our linear baseline environment, evaluating performance using **MAE, RMSE, and $R^2$**:

* **Standardized OLS Linear Regression**: Training the classical linear model on features processed via `StandardScaler` to evaluate performance under a normally distributed feature space.
* **Normalized OLS Linear Regression**: Training the classical linear model on features processed via `MinMaxScaler` to evaluate performance under a strictly bounded [0, 1] feature space.


In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Linear_Regression")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr, y_tr_pred))
    mlflow.log_metric("rmse_train", np.sqrt(mean_squared_error(y_tr, y_tr_pred)))
    mlflow.log_metric("r2_train", r2_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("mae_test", mean_absolute_error(y_te, y_te_pred))
    mlflow.log_metric("rmse_test", np.sqrt(mean_squared_error(y_te, y_te_pred)))
    mlflow.log_metric("r2_test", r2_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# Tournament Loop
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"OLS_LinearReg_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        model = LinearRegression()
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("model_type", "OLS_LinearRegression")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("fit_intercept", model.fit_intercept)
        
        log_regression_metrics(model, X_train_scaled, y_train, X_test_scaled, y_test, duration)


## **Winner Run Selection**

### **Policy**
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying MAE/RMSE/R² decision rules, we require Train→Test gaps to remain small enough to indicate acceptable generalization. Runs with large gaps are disqualified.

### **Selection Criteria (priority order)**
1. **Generalization filter (mandatory):** disqualify runs with large Train→Test gaps.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — secondary check to avoid catastrophic errors.
4. **Priority 3 (10%): Acceptable R² (Test)** — quality check for explained variance.
5. **Tiebreaker: Lowest Fit Time** — used only when previous metrics are effectively tied.

### **Runs: Summary Table (Train / Test + Gaps)**

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time (s) | Gen. Filter | MAE Gap | RMSE Gap | R² Gap |
|---|---:|---:|---:|---:|---:|---:|---:|:--:|---:|---:|---:|
| OLS_LinearReg_Standardization | 0.39666 | 0.40386 | 0.69367 | 0.71129 | 0.99413 | 0.99387 | 0.20648 | PASS | +0.00720 | +0.01762 | -0.00026 |
| OLS_LinearReg_Normalization | 0.39666 | 0.40386 | 0.69367 | 0.71129 | 0.99413 | 0.99387 | 0.19760 | PASS | +0.00720 | +0.01762 | -0.00026 |

### **Generalization Check (Test − Train)**
- Both runs: MAE gap = +0.00720, RMSE gap = +0.01762, R² gap = −0.00026 → PASS (gaps small; acceptable generalization).

### **Step-by-Step Elimination**
**Step 1 — Apply generalization filter**
- Passing runs: OLS_LinearReg_Standardization, OLS_LinearReg_Normalization.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- Both candidates: MAE (Test) = 0.40386 (tie).

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- Both candidates: RMSE (Test) = 0.71129 (tie).

**Step 4 — Check Test R² (Priority 3 — 10%)**
- Both candidates: R² (Test) = 0.99387 (tie).

**Step 5 — Tiebreaker: Fit Time**
- OLS_LinearReg_Normalization: 0.19760 s ← **WINNER (faster)**
- OLS_LinearReg_Standardization: 0.20648 s

### **Final Decision**
**Winner: OLS_LinearReg_Normalization**

**Justification:** Both runs show identical predictive performance (MAE, RMSE, R²) and pass generalization checks; `OLS_LinearReg_Normalization` is selected by the fit-time tiebreaker as it trains marginally faster while maintaining equivalent generalization.

### **Winner Hyperparameters**

| Parameter | Value |
|---|---|
| **fit_intercept** | True |
| **scaler** | Normalization |

## **Overfitting / Underfitting Diagnosis**
- Train vs Test gaps are minimal for both runs; no evidence of overfitting or underfitting.
- Conclusion: OLS Linear Regression generalizes well; the normalization variant is operationally preferred due to slightly lower training time.